In [1]:
%load_ext autoreload
%autoreload 2
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
from estimation_fct import *
from graph_format import *
import copy

# Load model
from model import ModelClass

model = ModelClass() 

par = model.par
sol = model.sol
sim = model.sim

par.N_a = 4
par.N_k = 4
par.N_s = 4

par.speed = "FAST"

efterloen_options = [1, 0]
flexible_hours_options = [par.flexible_hours, "Fixed"]

shares_efterloen = [0.32390004, 1 - 0.32390004]
shares_inflexible = [0.57511653, 1 - 0.57511653]

def get_weighted_model(model, shares_efterloen, shares_inflexible):
    models = []

    def append_simulations(target_model, source_model):
        for key, value in target_model.sim.__dict__.items():
            source_value = getattr(source_model.sim, key)

            if isinstance(value, np.ndarray):
                # append across individuals
                if value.ndim == 1:
                    combined = np.concatenate([value, source_value], axis=0)
                else:
                    combined = np.concatenate([value, source_value], axis=0)

                setattr(target_model.sim, key, combined)
            else:
                # keep non-array fields as-is, or overwrite if needed
                setattr(target_model.sim, key, source_value)


    for index_1, efterloen in enumerate(efterloen_options):
        for index_2, flexible_hours in enumerate(flexible_hours_options):

            par.simN = int(50000 * shares_efterloen[index_1] * shares_inflexible[index_2])

            par.efter = efterloen
            par.flexible_hours = flexible_hours

            model.solve()
            model.simulate()

            models.append(copy.deepcopy(model))

    model_weighted = copy.deepcopy(models[0])

    for m in models[1:]:
        append_simulations(model_weighted, m)

    return model_weighted

In [3]:
model_og = copy.deepcopy(get_weighted_model(model, shares_efterloen, shares_inflexible))

par.first_retirement = 31

model_new = copy.deepcopy(get_weighted_model(model, shares_efterloen, shares_inflexible))